In [4]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 12
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

Daniel Lei
Turki 
Ruoxi
Grace

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

here is a test - hello there why is it not working

pls work

final

pull req



In [14]:

tickers_df = pd.read_csv('Tickers_Example.csv', header = None, dtype=str)

start_date = '2025-10-01'
end_date = '2026-10-01'

ticker_list = []
for ticker in tickers_df.iloc[:, 0].dropna():
    ticker = ticker.replace(' ', '').upper()
    # skip blanks any title rows and duplicates (e.g. RY.TO listed twice)
    if ticker == '' or ticker == 'TICKER' or ticker in ticker_list:
        continue

    ticker_list.append(ticker)

In [17]:
symbols = ticker_list + ['^GSPC', 'CAD=X']

close = pd.DataFrame()
volume = pd.DataFrame()
for attempt in range(3):
    missing = [t for t in symbols if t not in close.columns]
    if len(missing) == 0:
        break
    data = yf.download(missing, start=start_date, end=end_date, auto_adjust=False, progress=False)
    if data.empty:
        continue
    new_close = data['Close'].dropna(axis=1, how='all')
    close = pd.concat([close, new_close], axis=1)
    volume = pd.concat([volume, data['Volume'][new_close.columns]], axis=1)

print('No price data (invalid or delisted):', [t for t in ticker_list if t not in close.columns])

$RTN: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$MON: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found

5 Failed downloads:
['RTN', 'AGN', 'MON', 'CELG']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$MON: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found

5 Failed downloads:
['MON', 'AGN', 'RTN', 'CELG']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$MON: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$BK: possibly delisted; no price data fou

No price data (invalid or delisted): ['AGN', 'BK', 'CELG', 'MON', 'RTN']


In [18]:
stock_volume = volume.drop(columns=['^GSPC', 'CAD=X'], errors='ignore')

# months with fewer than 18 trading days are ignored
days_per_month = stock_volume.groupby(stock_volume.index.to_period('M')).transform('count')
stock_volume = stock_volume.where(days_per_month >= 18)

mean_volumes = stock_volume.mean()
filtered_tickers = mean_volumes[mean_volumes >= 5000].index.tolist()
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [19]:
allowed_exchanges = ['NMS', 'NGM', 'NCM', 'NYQ', 'NYS', 'ASE', 'PCX', 'BTS', 'BATS', 'NAS', 'TOR', 'VAN', 'CNQ', 'NEO']

info_stocks = {}
for ticker in filtered_tickers:
    info = yf.Ticker(ticker).get_info()
    if info.get('quoteType') != 'EQUITY':
        print(f'Removed (not a stock): {ticker}')
    elif info.get('currency') not in ['CAD', 'USD']:
        print(f'Removed (currency is {info.get("currency")}): {ticker}')
    elif info.get('exchange') not in allowed_exchanges:
        print(f'Removed (not listed in Canada/US, exchange is {info.get("exchange")}): {ticker}')
    else:
        info_stocks[ticker] = {'Name': info.get('shortName', ticker),
                               'Sector': info.get('sector') or 'Unknown',
                               'Currency': info['currency']}

info_stocks = pd.DataFrame(info_stocks).T
filtered_tickers = info_stocks.index.tolist()
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [20]:
seen = {}
for t in filtered_tickers.copy():
    key = info_stocks.loc[t, 'Name'].lower().replace('.', '').replace(',', '').strip()
    if key not in seen:
        seen[key] = t
    elif mean_volumes[t] > mean_volumes[seen[key]]:
        print(f'Removed (second listing of the same company): {seen[key]}')
        filtered_tickers.remove(seen[key])
        seen[key] = t
    else:
        print(f'Removed (second listing of the same company): {t}')
        filtered_tickers.remove(t)

info_stocks = info_stocks.loc[filtered_tickers]
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [ ]:
temp_list = ticker_data['Close']
ticker_list = temp_list.columns.tolist()
ticker_list


['ABBV',
 'ABT',
 'ACN',
 'AGN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BK',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CELG',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MON',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RTN',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [22]:
fx = close['CAD=X'].ffill()
market = close['^GSPC'].ffill() * fx

prices = close[filtered_tickers].ffill()
for t in filtered_tickers:
    if info_stocks.loc[t, 'Currency'] == 'USD':
        prices[t] = prices[t] * fx

returns = prices.pct_change().iloc[1:]
market_returns = market.pct_change().iloc[1:]

beta = {}
for t in filtered_tickers:
    beta[t] = returns[t].cov(market_returns) / market_returns.var()
beta = pd.Series(beta)

standard_deviation = returns.std()
average_return = returns.mean()

pd.DataFrame({'Beta': beta, 'Daily std': standard_deviation, 'Avg daily return': average_return})

,Beta,Daily std,Avg daily return
AAPL,0.736774,0.015533,0.001223
ABBV,0.052005,0.016383,0.000471
ABT,0.132084,0.016980,-0.000947
ACN,0.337659,0.028184,-0.000625
AIG,0.102999,0.015198,-0.000074
AMZN,1.367508,0.021372,0.000768
AXP,0.998835,0.016921,-0.000082
BA,1.293161,0.021654,-0.000258
BAC,0.761664,0.014034,0.000449
BB.TO,1.406293,0.037817,0.003238


After removing all the unwanted stocks, get data for each of the remaining stocks that will help us determine if they are stable and optimal for us to add to our portfolio:
Standard deviation
Tells us how volatile/risky the stock is, and we will look for stocks that have lower standard deviations (close to 0) to reduce the volatility and risk of our portfolio
Beta to the S&P 500
Tells us how closely the stock moves with the market. We want to look for stocks that have lower betas (close to 0 in absolute value) because it will mean that the stock's movements aren't very correlated with the market, and don't particularly go up or down when the market moves
Expected Monthly Returns
Tells us what percentage return we should expect to see per month. We want to look for stocks that have lower expected monthly returns (close to 0 in absolute value) because we want a portfolio that gives as close to 0% returns as possible

| Signal | Why it points toward higher return | Weight |
|---|---|---|
| **Beta** (Chapter 15) | In a rising market, a beta-1.5 stock tends to move about 1.5× the market. CAPM: more market risk earns more expected return. | 40% |
| **12-1 month momentum** | Past-year winners (skipping the latest month, which tends to bounce back) have tended to keep winning (Jegadeesh & Titman, 1993). Research evidence, not from our slides. | 40% |
| **Revenue growth** (Yahoo) | Our client wants *growth*; fast-growing sales support a company's story. Yahoo's data is patchy, so it gets less weight and **a missing value counts as neutral**. | 20% |

**Selection:** top 12 by score, at most 4 per sector. **Weighting:** proportional to score (shifted so the lowest of the 12 is still positive),
then the 15% / 40% caps are applied and any excess is spread over the uncapped stocks.
**What we deliberately did *not* use:** dividends — the competition ignores dividends, so they earn nothing in this contest.

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.
Client A aims for the maximum return and is willing to take high risks. As expected returns cannot be observed, I have measured each stock against three factors and aggregated them using their z-scores (their distance from the mean of the entire set). Beta will account for 40%: stock with beta greater than 1 will move even more upwards than the movement of the market, and hence, the theory suggests that investors get higher compensation for market risk according to CAPM. Momentum over 12-1 month period will get 40%. 20% for revenue growth that corresponds to the growth strategy, although it is given less weight since the data for Yahoo is unreliable and the absence of a figure means a neutral score. I have purchased the top 12 based on their scores, where each sector is allowed a maximum of 4 stocks so that no single sector will be dominant. This was done intentionally since the contest gives recognition solely based on the best performing stock over five days, and with fewer stocks, there is greater variation in performance. Higher scores receive larger allocations with the stock limit of 15% and the sector limit of 40%. 

Client A is likely high-scoring (higher beta, growth, strong past year): so stocks like 
AMZN, SHOP.TO, QCOM, LLY, PYPL, AXP, CAT, BA, plus possibly C, BAC, TXN and ACN. BB.TO is very volatile and could score high or low depending on its momentum. The 4-per-sector limit will push the list to spread across tech, financials, industrials and healthcare.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.
If the market falls between Nov 30 and Dec 4, high-beta stocks will fall more than the market, so the beta tilt works against us. Momentum can reverse suddenly, and over only five days luck matters more than any signal. Up to 40% of the portfolio can sit in one sector, so a sector-specific shock would hurt. US stocks also lose value in CAD if the Canadian dollar strengthens.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.